# Zero-shot variant effect prediction with ESM-2

**Question:** Can a protein language model that has never seen lab data predict how single amino-acid
substitutions change protein function, and does a bigger model predict better?

**Approach:** Score every single substitution in a deep mutational scanning (DMS) assay from
[ProteinGym](https://proteingym.org) with ESM-2's masked-marginal log-likelihood ratio, then compare to the
measured fitness with Spearman ρ. Repeat for three model sizes (35M, 150M, 650M) and analyze where the model fails.

Runtime: **Runtime → Change runtime type → T4 GPU**. A full run takes roughly 5–10 minutes.

In [ ]:
!pip install -q transformers

import os, json, gc, zipfile, subprocess
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from transformers import AutoTokenizer, EsmForMaskedLM

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

## 1. Configuration

`BLAT_ECOLX_Stiffler_2015`: TEM-1 β-lactamase (286 aa), ~5,000 single mutants, fitness = ampicillin
resistance (growth). It is small, well studied, and contains only single mutants. To try another protein,
change `DMS_ID` to any ID in ProteinGym's `DMS_substitutions.csv`.

In [ ]:
DMS_ID = "BLAT_ECOLX_Stiffler_2015"

MODELS = {                       # name -> parameter count (millions)
    "facebook/esm2_t12_35M_UR50D": 35,
    "facebook/esm2_t30_150M_UR50D": 150,
    "facebook/esm2_t33_650M_UR50D": 650,
}

PG_VERSION = "v1.3"
DATA_DIR, FIG_DIR, RES_DIR = "data", "figures", "results"
for d in (DATA_DIR, FIG_DIR, RES_DIR):
    os.makedirs(d, exist_ok=True)

AAS = list("ACDEFGHIKLMNPQRSTVWY")

## 2. Download the assay

In [ ]:
ref_path = f"{DATA_DIR}/DMS_substitutions.csv"
if not os.path.exists(ref_path):
    subprocess.run(["wget", "-q", "-O", ref_path,
        "https://raw.githubusercontent.com/OATML-Markslab/ProteinGym/main/reference_files/DMS_substitutions.csv"], check=True)
ref = pd.read_csv(ref_path).set_index("DMS_id")
WT_SEQ = ref.loc[DMS_ID, "target_seq"]
print(DMS_ID, "| length:", len(WT_SEQ), "| assay:", ref.loc[DMS_ID, "selection_assay"])

dms_path = f"{DATA_DIR}/{DMS_ID}.csv"
if not os.path.exists(dms_path):
    zip_path = f"{DATA_DIR}/DMS_ProteinGym_substitutions.zip"   # ~1 GB, downloaded once
    subprocess.run(["wget", "-q", "-O", zip_path,
        f"https://marks.hms.harvard.edu/proteingym/ProteinGym_{PG_VERSION}/DMS_ProteinGym_substitutions.zip"], check=True)
    with zipfile.ZipFile(zip_path) as z:
        member = next(n for n in z.namelist() if n.endswith(f"{DMS_ID}.csv"))
        with open(dms_path, "wb") as f:
            f.write(z.read(member))
    os.remove(zip_path)

In [ ]:
def load_singles(path, wt_seq):
    df = pd.read_csv(path)
    df = df[~df["mutant"].str.contains(":")].copy()          # single substitutions only
    df["wt"] = df["mutant"].str[0]
    df["pos"] = df["mutant"].str[1:-1].astype(int)            # 1-based position in wt_seq
    df["mt"] = df["mutant"].str[-1]
    mismatch = [m for m, w, p in zip(df["mutant"], df["wt"], df["pos"]) if wt_seq[p - 1] != w]
    assert not mismatch, f"WT residue mismatch, e.g. {mismatch[:5]}"
    return df[["mutant", "wt", "pos", "mt", "DMS_score"]].reset_index(drop=True)

dms = load_singles(dms_path, WT_SEQ)
print(f"{len(dms)} single mutants across {dms['pos'].nunique()} positions")
dms.head()

## 3. Masked-marginal scoring

For each position *i*, mask it, run the model, and read the log-probability of every amino acid.
The score for mutation wt→mt at *i* is

$$ s = \log p(x_i = \text{mt} \mid x_{\setminus i}) - \log p(x_i = \text{wt} \mid x_{\setminus i}) $$

Negative scores mean the model thinks the mutant residue fits worse than the wild type.
One masked pass per position gives a full L × 20 saturation map, so every single mutant is scored at once.

In [ ]:
@torch.no_grad()
def masked_marginal_map(seq, model, tokenizer, batch_size=16, device=DEVICE):
    """Returns an (L, 20) array of log p(aa) - log p(wt) at each masked position."""
    enc = tokenizer(seq, return_tensors="pt")
    ids = enc["input_ids"][0]                                 # [<cls>, seq..., <eos>]
    L = len(seq)
    aa_ids = torch.tensor(tokenizer.convert_tokens_to_ids(AAS))
    wt_ids = ids[1:L + 1]
    out = torch.empty(L, len(AAS))
    for start in range(0, L, batch_size):
        positions = torch.arange(start, min(start + batch_size, L))
        batch = ids.repeat(len(positions), 1)
        batch[torch.arange(len(positions)), positions + 1] = tokenizer.mask_token_id   # +1 skips <cls>
        logits = model(input_ids=batch.to(device)).logits.float().cpu()
        logp = torch.log_softmax(logits[torch.arange(len(positions)), positions + 1], dim=-1)
        out[positions] = logp[:, aa_ids] - logp.gather(1, wt_ids[positions].unsqueeze(1))
    return out.numpy()

def score_mutants(dms, score_map):
    aa_index = {a: k for k, a in enumerate(AAS)}
    return np.array([score_map[p - 1, aa_index[m]] for p, m in zip(dms["pos"], dms["mt"])])

## 4. Run all three model sizes

In [ ]:
results, maps = {}, {}
for name, size in MODELS.items():
    tok = AutoTokenizer.from_pretrained(name)
    model = EsmForMaskedLM.from_pretrained(name).to(DEVICE).eval()
    if DEVICE == "cuda":
        model = model.half()
    maps[name] = masked_marginal_map(WT_SEQ, model, tok)
    dms[f"score_{size}M"] = score_mutants(dms, maps[name])
    rho = spearmanr(dms[f"score_{size}M"], dms["DMS_score"]).statistic
    results[f"{size}M"] = rho
    print(f"ESM-2 {size:>4}M  Spearman rho = {rho:.3f}")
    del model; gc.collect(); torch.cuda.empty_cache() if DEVICE == "cuda" else None

dms.to_csv(f"{RES_DIR}/{DMS_ID}_scores.csv", index=False)

### Uncertainty: bootstrap 95% CIs

Differences between model sizes only matter if they exceed sampling noise.

In [ ]:
rng = np.random.default_rng(0)
boot = {k: [] for k in results}
for _ in range(1000):
    idx = rng.integers(0, len(dms), len(dms))
    sample = dms.iloc[idx]
    for k in results:
        boot[k].append(spearmanr(sample[f"score_{k}"], sample["DMS_score"]).statistic)
ci = {k: (np.percentile(v, 2.5), np.percentile(v, 97.5)) for k, v in boot.items()}
for k in results:
    print(f"{k:>5}: rho = {results[k]:.3f}  (95% CI {ci[k][0]:.3f}-{ci[k][1]:.3f})")

## 5. Results plots

In [ ]:
best = max(results, key=results.get)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

sizes = list(results)
err = np.array([[results[k] - ci[k][0], ci[k][1] - results[k]] for k in sizes]).T
axes[0].bar(sizes, [results[k] for k in sizes], yerr=err, capsize=5, color="#4C72B0")
axes[0].set_xlabel("ESM-2 model size"); axes[0].set_ylabel("Spearman ρ vs. DMS")
axes[0].set_title("Zero-shot accuracy by model size")
for i, k in enumerate(sizes):
    axes[0].text(i, results[k] / 2, f"{results[k]:.2f}", ha="center", color="white", fontweight="bold")

axes[1].scatter(dms[f"score_{best}"], dms["DMS_score"], s=4, alpha=0.35, color="#4C72B0")
axes[1].set_xlabel(f"ESM-2 {best} masked-marginal score"); axes[1].set_ylabel("Measured DMS score")
axes[1].set_title(f"{DMS_ID}  (ρ = {results[best]:.2f}, n = {len(dms)})", fontsize=10)
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/model_size_and_scatter.png", dpi=200); plt.show()

In [ ]:
# Measured vs. predicted saturation maps (positions x amino acids), best model
measured = dms.pivot_table(index="mt", columns="pos", values="DMS_score").reindex(AAS)
predicted = pd.DataFrame(maps[[n for n, s in MODELS.items() if f"{s}M" == best][0]].T,
                         index=AAS, columns=range(1, len(WT_SEQ) + 1))[measured.columns]
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
for ax, mat, title in [(axes[0], measured, "Measured (DMS)"), (axes[1], predicted, f"ESM-2 {best} (zero-shot)")]:
    im = ax.imshow(mat.values, aspect="auto", cmap="RdBu", interpolation="nearest")
    ax.set_yticks(range(20)); ax.set_yticklabels(AAS, fontsize=6); ax.set_title(title)
    plt.colorbar(im, ax=ax, fraction=0.015)
axes[1].set_xlabel("Position (index into assayed positions)")
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/saturation_maps.png", dpi=200); plt.show()

## 6. Error analysis: where does the model fail?

Three cuts on the best model:
1. **By wild-type and mutant residue:** is accuracy uneven across amino-acid types?
2. **By position:** at which positions do the model's and the assay's sensitivity disagree most?
3. **Ranking failures:** which individual mutations does the model rank most wrongly?

In [ ]:
s = f"score_{best}"
by_wt = dms.groupby("wt").apply(lambda g: spearmanr(g[s], g["DMS_score"]).statistic, include_groups=False)
by_mt = dms.groupby("mt").apply(lambda g: spearmanr(g[s], g["DMS_score"]).statistic, include_groups=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.6), sharey=True)
for ax, ser, label in [(axes[0], by_wt, "wild-type residue"), (axes[1], by_mt, "mutant residue")]:
    ser = ser.reindex(AAS)
    ax.bar(ser.index, ser.values, color=["#C44E52" if v < results[best] - 0.1 else "#4C72B0" for v in ser.values])
    ax.axhline(results[best], ls="--", c="k", lw=1, label="overall ρ")
    ax.set_title(f"Spearman ρ grouped by {label}"); ax.set_xlabel(label)
axes[0].set_ylabel("Spearman ρ"); axes[0].legend()
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/error_by_residue.png", dpi=200); plt.show()

print("Weakest wild-type residues:", by_wt.sort_values().head(3).round(2).to_dict())
print("Weakest mutant residues:   ", by_mt.sort_values().head(3).round(2).to_dict())

In [ ]:
# Position-level sensitivity: mean effect of all substitutions at a position
pos = dms.groupby("pos").agg(wt=("wt", "first"), dms_mean=("DMS_score", "mean"), model_mean=(s, "mean"), n=("mt", "size"))
pos = pos[pos["n"] >= 10]
pos_rho = spearmanr(pos["model_mean"], pos["dms_mean"]).statistic
within = dms.groupby("pos").apply(lambda g: spearmanr(g[s], g["DMS_score"]).statistic if len(g) >= 10 else np.nan,
                                  include_groups=False).dropna()
print(f"Position-level rho (which positions are sensitive): {pos_rho:.3f}")
print(f"Median within-position rho (which substitution is worst at a position): {within.median():.3f}")

# Rank disagreement: model calls the position tolerant, assay says sensitive (and vice versa)
pos["rank_gap"] = pos["model_mean"].rank(pct=True) - pos["dms_mean"].rank(pct=True)
print("\nPositions the model thinks are TOLERANT but the assay says are SENSITIVE:")
print(pos.sort_values("rank_gap", ascending=False).head(8)[["wt", "dms_mean", "model_mean", "rank_gap"]].round(2))
print("\nPositions the model thinks are SENSITIVE but the assay says are TOLERANT:")
print(pos.sort_values("rank_gap").head(8)[["wt", "dms_mean", "model_mean", "rank_gap"]].round(2))

fig, ax = plt.subplots(figsize=(13, 3.2))
ax.plot(pos.index, pos["dms_mean"].rank(pct=True), label="DMS sensitivity rank", lw=1)
ax.plot(pos.index, pos["model_mean"].rank(pct=True), label=f"ESM-2 {best} rank", lw=1, alpha=0.8)
ax.set_xlabel("Position"); ax.set_ylabel("Percentile (low = sensitive)"); ax.legend(fontsize=8)
ax.set_title(f"Per-position mutational tolerance (position-level ρ = {pos_rho:.2f})")
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/position_tolerance.png", dpi=200); plt.show()

In [ ]:
# Individual mutations ranked most wrongly
dms["rank_gap"] = dms[s].rank(pct=True) - dms["DMS_score"].rank(pct=True)
print("Model says benign, assay says damaging:")
print(dms.sort_values("rank_gap", ascending=False).head(10)[["mutant", s, "DMS_score"]].round(2).to_string(index=False))
print("\nModel says damaging, assay says benign:")
print(dms.sort_values("rank_gap").head(10)[["mutant", s, "DMS_score"]].round(2).to_string(index=False))

## 7. Save a summary for the README

In [ ]:
summary = {
    "dms_id": DMS_ID,
    "n_single_mutants": int(len(dms)),
    "n_positions": int(dms["pos"].nunique()),
    "spearman_by_model": {k: round(float(v), 3) for k, v in results.items()},
    "spearman_95ci": {k: [round(float(a), 3), round(float(b), 3)] for k, (a, b) in ci.items()},
    "best_model": best,
    "position_level_rho": round(float(pos_rho), 3),
    "median_within_position_rho": round(float(within.median()), 3),
    "weakest_wt_residues": by_wt.sort_values().head(3).round(3).to_dict(),
    "weakest_mt_residues": by_mt.sort_values().head(3).round(3).to_dict(),
}
with open(f"{RES_DIR}/summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))